# Imports

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

sys.path.append(os.path.abspath("..\src"))

from config import CITIES, START_DATE, END_DATE, DAILY_VARIABLES
from ingestion import fetch_all_cities, fetch_forecast_all_cities

# Create output folders

In [2]:
raw_historical_dir = Path("../data/raw/historical")
raw_forecast_dir = Path("../data/raw/forecast")

raw_historical_dir.mkdir(parents=True, exist_ok=True)
raw_forecast_dir.mkdir(parents=True, exist_ok=True)

print("Historical folder:", raw_historical_dir.resolve())
print("Forecast folder:", raw_forecast_dir.resolve())

Historical folder: C:\Users\viktus\Desktop\for_labs\module5\m5-project-weather-pipeline\data\raw\historical
Forecast folder: C:\Users\viktus\Desktop\for_labs\module5\m5-project-weather-pipeline\data\raw\forecast


# Quick config check

In [3]:
print("Cities:")
for city in CITIES:
    print(city)

print("\nDate range:")
print(START_DATE, "→", END_DATE)

print("\nDaily variables:")
print(DAILY_VARIABLES)

Cities:
{'name': 'Baku', 'latitude': 40.41, 'longitude': 49.87}
{'name': 'Lankaran', 'latitude': 38.75, 'longitude': 48.85}
{'name': 'Guba', 'latitude': 41.36, 'longitude': 48.51}
{'name': 'Gabala', 'latitude': 40.58, 'longitude': 47.5}
{'name': 'Shaki', 'latitude': 41.11, 'longitude': 47.1}

Date range:
2020-01-01 → 2025-12-31

Daily variables:
['temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration']


# Fetch historical data for all cities

In [4]:
historical_data = fetch_all_cities(
    cities_config=CITIES,
    start_date=START_DATE,
    end_date=END_DATE,
    variables=DAILY_VARIABLES,
)

Fetching historical data for Baku...
Baku: 2192 historical rows
Fetching historical data for Lankaran...
Lankaran: 2192 historical rows
Fetching historical data for Guba...
Guba: 2192 historical rows
Fetching historical data for Gabala...
Gabala: 2192 historical rows
Fetching historical data for Shaki...
Shaki: 2192 historical rows


# Inspect historical data

In [5]:
for city_name, df in historical_data.items():
    print(f"\nCity: {city_name}")
    print(f"Rows: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print(f"Date range: {df['time'].min()} → {df['time'].max()}")
    print("Missing values:")
    print(df.isna().sum())


City: Baku
Rows: 2192
Columns: ['time', 'temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration', 'city']
Date range: 2020-01-01 00:00:00 → 2025-12-31 00:00:00
Missing values:
time                         0
temperature_2m_max           0
precipitation_sum            0
wind_speed_10m_max           0
relative_humidity_2m_mean    0
cloud_cover_mean             0
apparent_temperature_max     0
sunshine_duration            0
city                         0
dtype: int64

City: Lankaran
Rows: 2192
Columns: ['time', 'temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration', 'city']
Date range: 2020-01-01 00:00:00 → 2025-12-31 00:00:00
Missing values:
time                         0
temperature_2m_max           0
precipitation_sum            0
wind_speed_10m_max           0
relative_humidit

# Save historical raw files

In [15]:
for city_name, df in historical_data.items():
    file_path = raw_historical_dir / f"{city_name.lower()}_2020_2025.parquet"
    df.to_parquet(file_path, index=False)
    print(f"Saved historical file: {file_path}")

Saved historical file: ..\data\raw\historical\baku_2020_2025.parquet
Saved historical file: ..\data\raw\historical\lankaran_2020_2025.parquet
Saved historical file: ..\data\raw\historical\guba_2020_2025.parquet
Saved historical file: ..\data\raw\historical\gabala_2020_2025.parquet
Saved historical file: ..\data\raw\historical\shaki_2020_2025.parquet


# Verify saved historical files

In [7]:
for file in sorted(raw_historical_dir.glob("*.parquet")):
    print(f"\nFile: {file.name}")
    df_check = pd.read_parquet(file)
    print(df_check.head(3))
    print(df_check.shape)


File: baku_2020_2025.parquet
        time  temperature_2m_max  temperature_2m_min  precipitation_sum  \
0 2020-01-01                11.4                 4.4                0.0   
1 2020-01-02                 8.6                 2.7                0.3   
2 2020-01-03                 7.6                 3.4                1.4   

   wind_speed_10m_max  apparent_temperature_max  relative_humidity_2m_mean  \
0                11.4                       8.8                         88   
1                33.0                       3.2                         86   
2                21.7                       3.3                         84   

   weather_code  city  
0             3  Baku  
1            51  Baku  
2            51  Baku  
(2192, 9)

File: baku_2021_2025.parquet
        time  temperature_2m_max  precipitation_sum  wind_speed_10m_max  \
0 2020-01-01                11.4                0.0                11.4   
1 2020-01-02                 8.6                0.0                33.

# Fetch forecast data for all cities

In [8]:
forecast_data = fetch_forecast_all_cities(
    cities_config=CITIES,
    variables=DAILY_VARIABLES,
)

Fetching forecast data for Baku...
Baku: 7 forecast rows
Fetching forecast data for Lankaran...
Lankaran: 7 forecast rows
Fetching forecast data for Guba...
[Guba] Attempt 1 failed: ('Connection aborted.', ConnectionResetError(10054, 'Удаленный хост принудительно разорвал существующее подключение', None, 10054, None))
Guba: 7 forecast rows
Fetching forecast data for Gabala...
Gabala: 7 forecast rows
Fetching forecast data for Shaki...
Shaki: 7 forecast rows


# Inspect forecast data

In [9]:
for city_name, df in forecast_data.items():
    print(f"\nCity: {city_name}")
    print(f"Rows: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print(f"Date range: {df['time'].min()} → {df['time'].max()}")
    print("Missing values:")
    print(df.isna().sum())


City: Baku
Rows: 7
Columns: ['time', 'temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration', 'city']
Date range: 2026-04-24 00:00:00 → 2026-04-30 00:00:00
Missing values:
time                         0
temperature_2m_max           0
precipitation_sum            0
wind_speed_10m_max           0
relative_humidity_2m_mean    0
cloud_cover_mean             0
apparent_temperature_max     0
sunshine_duration            0
city                         0
dtype: int64

City: Lankaran
Rows: 7
Columns: ['time', 'temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration', 'city']
Date range: 2026-04-24 00:00:00 → 2026-04-30 00:00:00
Missing values:
time                         0
temperature_2m_max           0
precipitation_sum            0
wind_speed_10m_max           0
relative_humidity_2m_m

# Save forecast raw files

In [10]:
for city_name, df in forecast_data.items():
    file_path = raw_forecast_dir / f"{city_name.lower()}_forecast.parquet"
    df.to_parquet(file_path, index=False)
    print(f"Saved forecast file: {file_path}")

Saved forecast file: ..\data\raw\forecast\baku_forecast.parquet
Saved forecast file: ..\data\raw\forecast\lankaran_forecast.parquet
Saved forecast file: ..\data\raw\forecast\guba_forecast.parquet
Saved forecast file: ..\data\raw\forecast\gabala_forecast.parquet
Saved forecast file: ..\data\raw\forecast\shaki_forecast.parquet


# Verify saved forecast files

In [11]:
for file in sorted(raw_forecast_dir.glob("*.parquet")):
    print(f"\nFile: {file.name}")
    df_check = pd.read_parquet(file)
    print(df_check.head(3))
    print(df_check.shape)


File: baku_forecast.parquet
        time  temperature_2m_max  precipitation_sum  wind_speed_10m_max  \
0 2026-04-24                15.6                0.0                13.4   
1 2026-04-25                14.1                0.0                34.3   
2 2026-04-26                15.0                0.0                23.2   

   relative_humidity_2m_mean  cloud_cover_mean  apparent_temperature_max  \
0                         82                67                      15.7   
1                         80                76                      11.0   
2                         81                89                      12.1   

   sunshine_duration  city  
0           37310.84  Baku  
1           32409.44  Baku  
2           44290.19  Baku  
(7, 9)

File: gabala_forecast.parquet
        time  temperature_2m_max  precipitation_sum  wind_speed_10m_max  \
0 2026-04-24                21.4                0.0                35.3   
1 2026-04-25                20.3                0.0          

# Combine historical data into one dataframe

In [12]:
historical_all = pd.concat(historical_data.values(), ignore_index=True)
historical_all = historical_all.sort_values(["city", "time"]).reset_index(drop=True)

print(historical_all.shape)
print(historical_all.head())

(10960, 9)
        time  temperature_2m_max  precipitation_sum  wind_speed_10m_max  \
0 2020-01-01                11.4                0.0                11.4   
1 2020-01-02                 8.6                0.0                33.0   
2 2020-01-03                 7.6                1.7                24.1   
3 2020-01-04                 8.4                0.2                22.7   
4 2020-01-05                 7.7                2.6                12.6   

   relative_humidity_2m_mean  cloud_cover_mean  apparent_temperature_max  \
0                         90                30                       8.8   
1                         88                50                       3.2   
2                         83                96                       3.3   
3                         83                98                       4.6   
4                         91               100                       6.4   

   sunshine_duration  city  
0           31885.13  Baku  
1           30162.88  B

# Combine forecast data into one dataframe

In [13]:
forecast_all = pd.concat(forecast_data.values(), ignore_index=True)
forecast_all = forecast_all.sort_values(["city", "time"]).reset_index(drop=True)

print(forecast_all.shape)
print(forecast_all.head())

(35, 9)
        time  temperature_2m_max  precipitation_sum  wind_speed_10m_max  \
0 2026-04-24                15.6                0.0                13.4   
1 2026-04-25                14.1                0.0                34.3   
2 2026-04-26                15.0                0.0                23.2   
3 2026-04-27                18.6                0.1                43.1   
4 2026-04-28                14.0                0.3                52.7   

   relative_humidity_2m_mean  cloud_cover_mean  apparent_temperature_max  \
0                         82                67                      15.7   
1                         80                76                      11.0   
2                         81                89                      12.1   
3                         82                94                      19.3   
4                         61                94                       6.9   

   sunshine_duration  city  
0           37310.84  Baku  
1           32409.44  Baku

# Final raw-data sanity checks

In [14]:
print("Historical duplicate rows:", historical_all.duplicated().sum())
print("Forecast duplicate rows:", forecast_all.duplicated().sum())

print("\nHistorical date range:", historical_all["time"].min(), "→", historical_all["time"].max())
print("Forecast date range:", forecast_all["time"].min(), "→", forecast_all["time"].max())

print("\nHistorical columns:")
print(historical_all.columns.tolist())

print("\nForecast columns:")
print(forecast_all.columns.tolist())

Historical duplicate rows: 0
Forecast duplicate rows: 0

Historical date range: 2020-01-01 00:00:00 → 2025-12-31 00:00:00
Forecast date range: 2026-04-24 00:00:00 → 2026-04-30 00:00:00

Historical columns:
['time', 'temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration', 'city']

Forecast columns:
['time', 'temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration', 'city']


# DuckDB Loading

# Data Quality Checks